# 04. 시리얼 통신 & 전체 파이프라인 통합

마지막으로, 앞서 배운 인지·판단 결과를 **하나의 제어 명령**으로 묶어 아두이노로 보내는 과정을 다룹니다.
- `protocol_convert_func_lib.convert_serial_message` — 조향/속도 → 시리얼 문자열
- `motion_planner_node` 의 **우선순위 판단 로직**(장애물 > 빨간불 > 주행)을 재현


## 공통 설정 실행
아래 셀은 **모든 노트북 맨 위에서 한 번 실행**합니다. 프로젝트 루트를 찾고, 실제 코드가 있는 `lib` 폴더들을 import 경로에 등록하며, 이미지 표시 헬퍼(`show`, `show_row`)를 정의합니다.


In [1]:
# === 공통 설정: 프로젝트 루트 찾기 + 라이브러리 경로 등록 ===
import sys, math
from pathlib import Path
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm

# 그래프 한글 라벨이 깨지지 않도록 한글 폰트 지정 (Windows=맑은고딕 / Linux=나눔고딕)
for _f in [r'C:\Windows\Fonts\malgun.ttf', '/usr/share/fonts/truetype/nanum/NanumGothic.ttf']:
    try:
        _fm.fontManager.addfont(_f)
        matplotlib.rcParams['font.family'] = _fm.FontProperties(fname=_f).get_name()
        break
    except Exception:
        pass
matplotlib.rcParams['axes.unicode_minus'] = False

def find_root(start):
    p = Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'camera_perception_pkg').exists():
            return cand
    raise RuntimeError('ros3_ws 루트를 찾지 못했습니다. 노트북을 education_notebooks/ 안에서 실행하세요.')

ROOT = find_root(Path.cwd())
print('프로젝트 루트:', ROOT)

# 각 패키지의 lib 경로를 import 경로에 등록
CAM_LIB   = ROOT/'src'/'camera_perception_pkg'/'camera_perception_pkg'/'lib'
LIDAR_LIB = ROOT/'src'/'lidar_perception_pkg'/'lidar_perception_pkg'/'lib'
DEC_LIB   = ROOT/'src'/'decision_making_pkg'/'decision_making_pkg'/'lib'
SER_LIB   = ROOT/'src'/'serial_communication_pkg'/'serial_communication_pkg'/'lib'
SAMPLES   = CAM_LIB/'Collected_Datasets'/'sample_dataset'
for p in (CAM_LIB, LIDAR_LIB, DEC_LIB, SER_LIB):
    sys.path.insert(0, str(p))

# Jupyter/터미널 어디서든 이미지를 인라인으로 보기 위한 헬퍼
def show(img, title='', cmap=None, figsize=(6,4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # OpenCV는 BGR, matplotlib은 RGB
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.tight_layout(); plt.show()

def show_row(imgs, titles=None, cmap=None, figsize=(14,4)):
    n = len(imgs); titles = titles or ['']*n
    plt.figure(figsize=figsize)
    for i,(im,t) in enumerate(zip(imgs, titles)):
        ax = plt.subplot(1, n, i+1)
        if im.ndim == 3: ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else: ax.imshow(im, cmap=cmap or 'gray')
        ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show()

print('OpenCV', cv2.__version__, '| NumPy', np.__version__)


프로젝트 루트: C:\Users\asus\Documents\workspace\ros3_ws
OpenCV 5.0.0 | NumPy 2.4.6


## 1) 시리얼 프로토콜 — `convert_serial_message`
아두이노는 문자열 명령을 받습니다. 원본 함수는 아주 단순합니다:
```python
def convert_serial_message(steering, left_speed, right_speed):
    message = f"s{steering}l{left_speed}r{right_speed}\n"
    return message
```
`s`=조향, `l`=좌측속도, `r`=우측속도. 아두이노(`driving.ino`)가 이 문자를 파싱합니다.


In [2]:
from protocol_convert_func_lib import convert_serial_message

print(repr(convert_serial_message(steering=7, left_speed=100, right_speed=100)))
print(repr(convert_serial_message(steering=-7, left_speed=100, right_speed=100)))
print(repr(convert_serial_message(steering=0, left_speed=0, right_speed=0)))   # 정지


's7l100r100\n'
's-7l100r100\n'
's0l0r0\n'


## 2) motion_planner 판단 로직 재현
`motion_planner_node.timer_callback()` 의 실제 **우선순위**:
1. **라이다 장애물 감지 → 즉시 정지** (steering=0, 속도=0)
2. 그 외 **빨간 신호등(가까이) → 정지**
3. 그 외 → **경로 기울기로 조향**, 기본 속도 100

아래는 이 규칙을 그대로 옮긴 통합 판단 함수입니다.


In [3]:
import decision_making_function_lib as dm

def motion_plan(path, lidar_obstacle, traffic_light):
    # 1순위: 장애물
    if lidar_obstacle is True:
        return 0, 0, 0, '장애물 감지 -> 정지'
    # 2순위: 빨간불
    if traffic_light == 'Red':
        return 0, 0, 0, '빨간불 -> 정지'
    # 3순위: 주행
    if path is None or len(path) < 10:
        steer = 0
    else:
        s = dm.calculate_slope_between_points(path[-10], path[-1])
        steer = 7 if (s != 'inf' and s > 0) else (-7 if (s != 'inf' and s < 0) else 0)
    return steer, 100, 100, '정상 주행'

# 예시 경로 (02번 노트북과 동일 방식)
from scipy.interpolate import CubicSpline
def make_path(tg, car=(320,179)):
    xs=[p[0] for p in tg]+[car[0]]; ys=[p[1] for p in tg]+[car[1]]
    ys,xs = zip(*sorted(zip(ys,xs)))
    cs=CubicSpline(ys,xs,bc_type='natural'); yn=np.linspace(min(ys),max(ys),100)
    return list(zip(cs(yn), yn))

path = make_path([(300,40),(330,80),(370,120),(410,160)])   # 오른쪽 커브
steer,ls,rs,reason = motion_plan(path, lidar_obstacle=False, traffic_light='Green')
print(reason, '->', convert_serial_message(steer, ls, rs).strip())



 ____    ___   ____    ____  
|  _ \  / _ \ / ___|  |___ \ 
| |_) || | | |\___ \    __) |
|  _ < | |_| | ___) |  / __/ 
|_| \_\ \___/ |____/  |_____|
                             
    _            _                 
   / \    _   _ | |_   ___         
  / _ \  | | | || __| / _ \  _____ 
 / ___ \ | |_| || |_ | (_) ||_____|
/_/   \_\ \__,_| \__| \___/        
                                   
__     __       _      _        _       
\ \   / /  ___ | |__  (_)  ___ | |  ___ 
 \ \ / /  / _ \| '_ \ | | / __|| | / _   \ V /  |  __/| | | || || (__ | ||  __/
   \_/    \___||_| |_||_| \___||_| \___|  


ROS2 기반 자율주행 설계 및 구현
Sungkyunkwan University Automation Lab.
------------------Authors------------------
Hyeong-Keun Hong <whaihong@g.skku.edu>
Jinsun Lee <with23skku@g.skku.edu>
Siwoo Lee <edenlee@g.skku.edu>
Jae-Wook Jeon <jwjeon@skku.edu>
------------------------------------------


정상 주행 -> s7l100r100


## 3) 여러 시나리오 시뮬레이션
같은 경로라도 **장애물/신호등 상황**에 따라 최종 시리얼 명령이 어떻게 바뀌는지 한눈에 봅니다.


In [4]:
scenarios = [
    ('평상시 주행(초록불)',        False, 'Green'),
    ('빨간불',                    False, 'Red'),
    ('장애물 감지',               True,  'Green'),
    ('장애물 + 빨간불(장애물 우선)', True,  'Red'),
]
print(f"{'시나리오':22s} | {'판단':14s} | 시리얼 명령")
print('-'*70)
for name, obs, tl in scenarios:
    st, ls, rs, why = motion_plan(path, obs, tl)
    msg = convert_serial_message(st, ls, rs).strip()
    print(f'{name:22s} | {why:14s} | {msg}')


시나리오                   | 판단             | 시리얼 명령
----------------------------------------------------------------------
평상시 주행(초록불)            | 정상 주행          | s7l100r100
빨간불                    | 빨간불 -> 정지      | s0l0r0
장애물 감지                 | 장애물 감지 -> 정지   | s0l0r0
장애물 + 빨간불(장애물 우선)      | 장애물 감지 -> 정지   | s0l0r0


## 4) 전 과정 한 번에 (이미지 → 시리얼)
01~04에서 배운 것을 하나로 연결합니다: **카메라 이미지 → 차선 기울기 → 조향 → 시리얼 명령**.
(신호등/라이다는 상황 값으로 주입)


In [5]:
import camera_perception_func_lib as cam

def full_pipeline(image_path, lidar_obstacle=False, traffic_light='Green'):
    img = cv2.imread(str(image_path))
    # --- 인지: 차선 기울기 ---
    src=[[238,316],[402,313],[501,476],[155,476]]
    h,w=img.shape[:2]; dst=[[round(w*.3),0],[round(w*.7),0],[round(w*.7),h],[round(w*.3),h]]
    roi = cam.roi_rectangle_below(cam.bird_convert(img,src,dst), 300)
    _, mask = cv2.threshold(cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY),150,255,cv2.THRESH_BINARY)
    angle = cam.dominant_gradient(mask, theta_limit=45)
    # --- 인지 결과를 경로처럼 사용: 기울기 부호로 간단 경로 생성 ---
    tg = [(320+int(angle*k), 40+40*k) for k in range(4)]
    path = make_path(tg)
    # --- 판단 + 제어 ---
    st, ls, rs, why = motion_plan(path, lidar_obstacle, traffic_light)
    msg = convert_serial_message(st, ls, rs).strip()
    print(f'{Path(image_path).name}: 차선기울기 {angle:.1f}도 | {why} | 시리얼 = {msg}')
    return msg

full_pipeline(SAMPLES/'sample01.png', lidar_obstacle=False, traffic_light='Green')
full_pipeline(SAMPLES/'sample01.png', lidar_obstacle=True,  traffic_light='Green')



 ____    ___   ____    ____  
|  _ \  / _ \ / ___|  |___ \ 
| |_) || | | |\___ \    __) |
|  _ < | |_| | ___) |  / __/ 
|_| \_\ \___/ |____/  |_____|
                             
    _            _                 
   / \    _   _ | |_   ___         
  / _ \  | | | || __| / _ \  _____ 
 / ___ \ | |_| || |_ | (_) ||_____|
/_/   \_\ \__,_| \__| \___/        
                                   
__     __       _      _        _       
\ \   / /  ___ | |__  (_)  ___ | |  ___ 
 \ \ / /  / _ \| '_ \ | | / __|| | / _   \ V /  |  __/| | | || || (__ | ||  __/
   \_/    \___||_| |_||_| \___||_| \___|  


ROS2 기반 자율주행 설계 및 구현
Sungkyunkwan University Automation Lab.
------------------Authors------------------
Hyeong-Keun Hong <whaihong@g.skku.edu>
Jinsun Lee <with23skku@g.skku.edu>
Siwoo Lee <edenlee@g.skku.edu>
Jae-Wook Jeon <jwjeon@skku.edu>
------------------------------------------
sample01.png: 차선기울기 1.0도 | 정상 주행 | 시리얼 = s7l100r100
sample01.png: 차선기울기 1.0도 | 장애물 감지 -> 정지 | 시리얼 = s0l0r0


's0l0r0'

---
## 🎯 미션 1 — 노란불 처리 추가
현재 `motion_plan` 은 빨간불만 정지합니다. **노란불(Yellow)** 이면 속도를 절반(50)으로 줄이는
규칙을 추가한 `motion_plan_v2` 를 완성하세요.


In [6]:
def motion_plan_v2(path, lidar_obstacle, traffic_light):
    if lidar_obstacle is True:
        return 0,0,0,'장애물 -> 정지'
    if traffic_light == 'Red':
        return 0,0,0,'빨간불 -> 정지'
    # TODO: traffic_light == 'Yellow' 이면 속도 50으로 서행 (조향은 주행과 동일 계산)
    ...


**▶ 정답 예시**


In [7]:
def motion_plan_v2(path, lidar_obstacle, traffic_light):
    if lidar_obstacle is True: return 0,0,0,'장애물 -> 정지'
    if traffic_light == 'Red':  return 0,0,0,'빨간불 -> 정지'
    s = dm.calculate_slope_between_points(path[-10], path[-1]) if len(path)>=10 else 0
    steer = 7 if (s!='inf' and s>0) else (-7 if (s!='inf' and s<0) else 0)
    spd = 50 if traffic_light=='Yellow' else 100
    return steer, spd, spd, ('노란불 -> 서행' if traffic_light=='Yellow' else '정상 주행')

for tl in ['Green','Yellow','Red']:
    st,ls,rs,why = motion_plan_v2(path, False, tl)
    print(f'{tl:6s}: {why:14s} -> {convert_serial_message(st,ls,rs).strip()}')


Green : 정상 주행          -> s7l100r100
Yellow: 노란불 -> 서행      -> s7l50r50
Red   : 빨간불 -> 정지      -> s0l0r0


## 🎯 미션 2 — 나만의 시나리오
장애물이 특정 시각에 나타났다 사라지는 **시간 시퀀스**를 만들어, 매 스텝의 시리얼 명령을 출력하세요.
(힌트: 03번의 `StabilityDetector` 를 붙이면 더 현실적입니다.)


In [8]:
# TODO: obstacle_timeline 을 만들고 반복문으로 매 스텝 motion_plan 실행
obstacle_timeline = [False,False,True,True,True,False,False]
for t, obs in enumerate(obstacle_timeline):
    st,ls,rs,why = motion_plan(path, obs, 'Green')
    print(f't={t}: {why:14s} -> {convert_serial_message(st,ls,rs).strip()}')


t=0: 정상 주행          -> s7l100r100
t=1: 정상 주행          -> s7l100r100
t=2: 장애물 감지 -> 정지   -> s0l0r0
t=3: 장애물 감지 -> 정지   -> s0l0r0
t=4: 장애물 감지 -> 정지   -> s0l0r0
t=5: 정상 주행          -> s7l100r100
t=6: 정상 주행          -> s7l100r100


---
## 🎉 전체 실습 완료!
인지(카메라·라이다) → 판단(경로·우선순위) → 제어(시리얼)의 자율주행 3단계를 실제 코드로 모두 다뤘습니다.

**다음 단계 (심화):**
- 실제 ROS2 환경(Linux)에서 `ros2 launch launch_pkg main.launch.py` 로 전체 노드를 띄워보기
- `best.pt`(YOLOv8) 모델로 실제 차선·객체 세그멘테이션 돌려보기 (`ultralytics` 필요)
- `driving_simulation.mp4` 영상에 파이프라인을 프레임 단위로 적용해 보기
